In [ ]:
pip install PuLP

Note: you may need to restart the kernel to use updated packages.


In [ ]:
pip install pandas

Note: you may need to restart the kernel to use updated packages.


In [ ]:
from pulp import *
import pandas as pd
import pulp

In [ ]:
#Data Setup

#origins with supply
origins = {'SAM': 1050, 'SEA': 30, 'EUR': 10000, 'ONO': 100}

#destination with demand
destinations = {'CAL': 171, 'WIN': 64, 'OTT': 127, 'MTL': 184, 'HAL': 64, 'OND': 381, 'VAD': 191}

#Hubs
hubs = ['ONH', 'MIA', 'VAH']

#cost: origin -> hub
cost_origin_hub = pd.DataFrame({'ONH': [14, 35, 30, 0], 'MIA': [9, 20, 15, 10000], 'VAH': [20, 35, 35, 15]}
                               , index=['SAM', 'SEA', 'EUR', 'ONO'])

#cost: hub -> destination
cost_hub_dest = pd.DataFrame({'CAL': [14, 10, 7], 'WIN': [13, 10, 8], 'OTT': [3.5, 7.5, 14], 'MTL': [3.5, 7.5, 14], 'HAL': [4, 10, 22], 'OND': [0, 5, 14], 'VAD': [15, 10, 0]}
                             , index=['ONH', 'MIA', 'VAH'])

In [ ]:
#Initialize the model

from pulp.constants import LpMinimize
model = pulp.LpProblem("Transshipment", LpMinimize)

# Decision Variables
x = pulp.LpVariable.dicts("x", (origins, hubs), lowBound=0, cat='Continuous')
y = pulp.LpVariable.dicts("y", (hubs, destinations), lowBound=0, cat='Continuous')

In [ ]:
model += (pulp.lpSum(cost_origin_hub.loc[i, j] * x[i][j] for i in origins for j in hubs if pd.notna(cost_origin_hub.loc[i, j])) + pulp.lpSum(cost_hub_dest.loc[j, k] * y[j][k]
               for j in hubs for k in destinations)
), "Total_Cost"

In [ ]:
# Constraints

# Supply at origins
for i in origins:
    if origins[i] != float('inf'):
        model += pulp.lpSum(x[i][j] for j in hubs if pd.notna(cost_origin_hub.loc[i, j])) <= origins[i], f"Supply_{i}"

# Demand at destinations
for k in destinations:
    model += pulp.lpSum(y[j][k] for j in hubs) == destinations[k], f"Demand_{k}"

# Flow balance at hubs
for j in hubs:
    inflow = pulp.lpSum(x[i][j] for i in origins if pd.notna(cost_origin_hub.loc[i, j]))
    outflow = pulp.lpSum(y[j][k] for k in destinations)
    model += inflow == outflow, f"FlowBalance_{j}"

In [ ]:
#Solve and Display

_ = model.solve(pulp.PULP_CBC_CMD(msg=False))
print("Status:", pulp.LpStatus[model.status])
print("Optimal total weekly cost (CAD):", round(pulp.value(model.objective), 2))


Status: Optimal
Optimal total weekly cost (CAD): 18503.5


In [ ]:
# Display all flows with a nonzero value
flows_x = [(i, j, x[i][j].varValue)
           for i in origins for j in hubs
           if x[i][j].varValue is not None]

flows_y = [(j, k, y[j][k].varValue)
           for j in hubs for k in destinations
           if y[j][k].varValue is not None]

print("\n Origin -> Hub Flows")
for i, j, q in flows_x:
    print(f"{i:15} -> {j:12} : {q:.2f}")

print("\n Hub -> Destination Flows")
for j, k, q in flows_y:
    print(f"{j:12} -> {k:12} : {q:.2f}")


 Origin -> Hub Flows
SAM             -> ONH          : 345.00
SAM             -> MIA          : 705.00
SAM             -> VAH          : 0.00
SEA             -> ONH          : 0.00
SEA             -> MIA          : 0.00
SEA             -> VAH          : 0.00
EUR             -> ONH          : 0.00
EUR             -> MIA          : 32.00
EUR             -> VAH          : 0.00
ONO             -> ONH          : 100.00
ONO             -> MIA          : 0.00
ONO             -> VAH          : 0.00

 Hub -> Destination Flows
ONH          -> CAL          : 0.00
ONH          -> WIN          : 0.00
ONH          -> OTT          : 0.00
ONH          -> MTL          : 0.00
ONH          -> HAL          : 64.00
ONH          -> OND          : 381.00
ONH          -> VAD          : 0.00
MIA          -> CAL          : 171.00
MIA          -> WIN          : 64.00
MIA          -> OTT          : 127.00
MIA          -> MTL          : 184.00
MIA          -> HAL          : 0.00
MIA          -> OND          : 0.0

In [ ]:
# Sensitivity Analysis
print()
print("Sensitivity Analysis:")
MF_SA_df = pd.DataFrame([
    {
        'constraint': name,
        'slack': c.slack,
        'shadow price': c.pi
    }
    for name, c in model.constraints.items()
])
print(MF_SA_df)


Sensitivity Analysis:
         constraint   slack  shadow price
0        Supply_SAM    -0.0          -6.0
1        Supply_SEA    30.0           0.0
2        Supply_EUR  9968.0           0.0
3        Supply_ONO    -0.0         -20.0
4        Demand_CAL    -0.0          25.0
5        Demand_WIN    -0.0          25.0
6        Demand_OTT    -0.0          22.5
7        Demand_MTL    -0.0          22.5
8        Demand_HAL    -0.0          24.0
9        Demand_OND    -0.0          20.0
10       Demand_VAD    -0.0          25.0
11  FlowBalance_ONH    -0.0          20.0
12  FlowBalance_MIA    -0.0          15.0
13  FlowBalance_VAH    -0.0          25.0
